# Modelagem inicial das faixas de atraso

Este notebook inicia a previsão da faixa de atraso de chegada. A divisão dos dados respeita a ordem temporal: meses antigos são usados para treinamento, meses seguintes para validação e os meses mais recentes ficam reservados para o teste final.

O alvo possui seis classes, de 0 (pontual ou antecipado) a 5 (atraso superior a 60 minutos).


## 1. Importar bibliotecas e localizar a base

Esta célula importa as bibliotecas usadas no notebook e procura o CSV de modelagem na raiz do projeto. O CSV é gerado pelo script scripts/preparar_modelagem.py.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, TargetEncoder

DATA_PATH = Path("data/modelagem_faixas_atraso.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("..") / "data" / "modelagem_faixas_atraso.csv"

FAIXAS = {
    0: "Pontual ou antecipado",
    1: "Atraso inferior a 15 min",
    2: "Atraso de 15 a 30 min",
    3: "Atraso superior a 30 até 45 min",
    4: "Atraso superior a 45 até 60 min",
    5: "Atraso superior a 60 min",
}


## 2. Carregar a base de modelagem

A base já contém o alvo faixa_atraso e as variáveis derivadas da partida prevista. A coluna data_referencia será usada somente para separar os períodos; ela não entrará como variável explicativa.


In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df["data_referencia"] = pd.to_datetime(df["data_referencia"])
print("Dimensões:", df.shape)
df.head()


## 3. Conferir a distribuição geral do alvo

Antes de treinar, verificamos quantos exemplos existem em cada faixa. A distribuição orienta a escolha das métricas e mostra por que a acurácia simples não é suficiente.


In [ ]:
distribuicao = (
    df["faixa_atraso"]
    .value_counts()
    .sort_index()
    .rename(index=FAIXAS)
    .to_frame("quantidade")
)
distribuicao["percentual"] = 100 * distribuicao["quantidade"] / len(df)
distribuicao


## 4. Conferir a estabilidade das classes ao longo do tempo

A tabela mensal permite verificar se alguma faixa desaparece ou muda muito de proporção. Essa verificação é importante antes da divisão temporal.


In [ ]:
mensal = (
    df.groupby(["mes_referencia", "faixa_atraso"], observed=False)
      .size()
      .unstack(fill_value=0)
)
mensal_percentual = mensal.div(mensal.sum(axis=1), axis=0) * 100
mensal_percentual.round(2)


## 5. Separar treino, validação e teste por data

Não embaralhamos os voos. A divisão é calculada a partir dos meses disponíveis no dataset, permitindo que o notebook continue funcionando quando novos meses forem acrescentados.

- os 3 meses mais recentes são reservados para teste;
- os 3 meses imediatamente anteriores são usados para validação;
- todos os meses restantes são usados para treinamento.

Assim, se o último mês for `2026-08`, o teste será de `2026-06` a `2026-08` e a validação será de `2026-03` a `2026-05`.

In [ ]:
MESES_VALIDACAO = 3
MESES_TESTE = 3

periodos = df["data_referencia"].dt.to_period("M")
meses_disponiveis = sorted(periodos.dropna().unique())
meses_necessarios = MESES_VALIDACAO + MESES_TESTE + 1

if len(meses_disponiveis) < meses_necessarios:
    raise ValueError(
        f"São necessários pelo menos {meses_necessarios} meses para criar "
        "treino, validação e teste."
    )

meses_teste = meses_disponiveis[-MESES_TESTE:]
meses_validacao = meses_disponiveis[-(MESES_VALIDACAO + MESES_TESTE):-MESES_TESTE]
meses_treino = meses_disponiveis[:-(MESES_VALIDACAO + MESES_TESTE)]

treino = df[periodos.isin(meses_treino)].copy()
validacao = df[periodos.isin(meses_validacao)].copy()
teste = df[periodos.isin(meses_teste)].copy()

print("Meses de treino:", [str(mes) for mes in meses_treino])
print("Meses de validação:", [str(mes) for mes in meses_validacao])
print("Meses de teste:", [str(mes) for mes in meses_teste])
print("Treino:", treino.shape)
print("Validação:", validacao.shape)
print("Teste:", teste.shape)


## 6. Selecionar as variáveis disponíveis no momento da previsão

As variáveis abaixo são conhecidas antes do resultado do voo. Não usamos horários reais, atrasos observados ou qualquer informação preenchida depois da chegada.


In [ ]:
categoricas = [
    "companhia_icao",
    "origem_icao",
    "destino_icao",
    "codigo_tipo_linha",
    "modelo_equipamento",
    "periodo_dia",
]
numericas = [
    "numero_assentos",
    "ano",
    "mes",
    "dia_semana",
    "hora_prevista",
    "fim_de_semana",
]

X_treino = treino[categoricas + numericas]
y_treino = treino["faixa_atraso"]
X_validacao = validacao[categoricas + numericas]
y_validacao = validacao["faixa_atraso"]
X_teste = teste[categoricas + numericas]
y_teste = teste["faixa_atraso"]


## 7. Criar o baseline da classe majoritária

Este modelo sempre prevê a faixa mais frequente no treinamento. Ele é simples, mas indispensável: qualquer modelo útil precisa superar esse resultado, especialmente em macro-F1 e balanced accuracy.


In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_treino, y_treino)
pred_baseline = baseline.predict(X_teste)

print("Acurácia:", accuracy_score(y_teste, pred_baseline))
print("Balanced accuracy:", balanced_accuracy_score(y_teste, pred_baseline))
print("Macro-F1:", f1_score(y_teste, pred_baseline, average="macro"))


## 8. Preparar a regressão logística multiclasses

Variáveis categóricas são convertidas para indicadores com one-hot encoding. Valores ausentes numéricos recebem a mediana e valores ausentes categóricos recebem a categoria mais frequente. As variáveis numéricas são padronizadas para facilitar a convergência do otimizador. O peso balanceado reduz o favorecimento da classe majoritária.


In [ ]:
preprocessador = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            Pipeline([
                ("imputacao", SimpleImputer(strategy="most_frequent")),
                ("one_hot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categoricas,
        ),
        (
            "numericas",
            Pipeline([
                ("imputacao", SimpleImputer(strategy="median")),
                ("escala", StandardScaler(with_mean=False)),
            ]),
            numericas,
        ),
    ]
)

modelo = Pipeline([
    ("preprocessamento", preprocessador),
    ("classificador", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="lbfgs",
    )),
])


## 9. Treinar o primeiro modelo

Esta célula pode levar algum tempo porque usa todos os voos do período de treinamento. O modelo aprende relações entre companhia, rota, horário e outras variáveis conhecidas antes da partida.


In [ ]:
modelo.fit(X_treino, y_treino)
pred_validacao = modelo.predict(X_validacao)
pred_teste = modelo.predict(X_teste)


## 10. Avaliar por classe

A tabela mostra precisão, recall e F1 de cada faixa. O recall das faixas graves é especialmente importante: ele indica quantos atrasos daquela severidade foram identificados.


In [ ]:
print("Validação")
print(classification_report(
    y_validacao,
    pred_validacao,
    labels=list(FAIXAS),
    target_names=list(FAIXAS.values()),
    zero_division=0,
))
print("Teste")
print(classification_report(
    y_teste,
    pred_teste,
    labels=list(FAIXAS),
    target_names=list(FAIXAS.values()),
    zero_division=0,
))


## 11. Comparar as métricas agregadas

Usamos acurácia apenas como referência. Balanced accuracy e macro-F1 dão o mesmo peso a todas as faixas, incluindo as menos frequentes.


In [ ]:
metricas = pd.DataFrame({
    "modelo": ["baseline", "regressao_logistica"],
    "acuracia_teste": [
        accuracy_score(y_teste, pred_baseline),
        accuracy_score(y_teste, pred_teste),
    ],
    "balanced_accuracy_teste": [
        balanced_accuracy_score(y_teste, pred_baseline),
        balanced_accuracy_score(y_teste, pred_teste),
    ],
    "macro_f1_teste": [
        f1_score(y_teste, pred_baseline, average="macro"),
        f1_score(y_teste, pred_teste, average="macro"),
    ],
})
metricas


## 12. Visualizar a matriz de confusão

A matriz mostra se os erros são próximos da faixa correta ou se o modelo confunde atrasos graves com voos pontuais. Como as classes são ordinais, erros distantes são mais preocupantes.


In [ ]:
def mostrar_matrizes(y_real, y_previsto, nome_modelo):
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    for ax, normalizacao, titulo, formato in [
        (axes[0], None, "Matriz absoluta", "d"),
        (axes[1], "true", "Matriz normalizada por classe real", ".2f"),
    ]:
        matriz = confusion_matrix(
            y_real, y_previsto, labels=list(FAIXAS), normalize=normalizacao
        )
        disp = ConfusionMatrixDisplay(
            confusion_matrix=matriz, display_labels=list(FAIXAS.values())
        )
        disp.plot(ax=ax, values_format=formato, xticks_rotation=45, colorbar=False)
        ax.set_title(titulo)
    fig.suptitle(f"Matrizes de confusão — {nome_modelo} — teste")
    plt.tight_layout()
    plt.show()

mostrar_matrizes(y_teste, pred_baseline, "baseline")
mostrar_matrizes(y_teste, pred_teste, "regressão logística")


## 13. Testar um Random Forest

O Random Forest usa várias árvores e pode capturar relações não lineares entre horários, rotas, companhias e equipamentos. Nesta primeira versão usamos codificação ordinal para as categorias e pesos balanceados.


In [ ]:
preprocessador_rf = Pipeline([
    ("imputacao", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])
modelo_rf = Pipeline([
    ("preprocessamento", preprocessador_rf),
    ("classificador", RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=10,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42,
    )),
])


## 14. Treinar e avaliar o Random Forest

A comparação usa as mesmas amostras de treino e teste dos modelos anteriores. Assim, a diferença de desempenho pode ser atribuída ao modelo e não à divisão dos dados.


In [ ]:
print("Rodando o modelo.fit(). Isso pode levar alguns minutos")
modelo_rf.fit(X_treino, y_treino)
pred_rf = modelo_rf.predict(X_teste)
print(classification_report(
    y_teste,
    pred_rf,
    labels=list(FAIXAS),
    target_names=list(FAIXAS.values()),
    zero_division=0,
))
print({
    "accuracy": accuracy_score(y_teste, pred_rf),
    "balanced_accuracy": balanced_accuracy_score(y_teste, pred_rf),
    "macro_f1": f1_score(y_teste, pred_rf, average="macro"),
})
mostrar_matrizes(y_teste, pred_rf, "random forest")


## 15. Atualizar a comparação dos modelos

O Random Forest será comparado com o baseline majoritário e a regressão logística usando balanced accuracy e macro-F1. A acurácia permanece apenas como referência complementar.


In [ ]:
metricas_rf = pd.DataFrame({
    "modelo": ["baseline", "regressao_logistica", "random_forest"],
    "balanced_accuracy_teste": [
        balanced_accuracy_score(y_teste, pred_baseline),
        balanced_accuracy_score(y_teste, pred_teste),
        balanced_accuracy_score(y_teste, pred_rf),
    ],
    "macro_f1_teste": [
        f1_score(y_teste, pred_baseline, average="macro"),
        f1_score(y_teste, pred_teste, average="macro"),
        f1_score(y_teste, pred_rf, average="macro"),
    ],
})
metricas_rf


## 16. Testar Target Encoding com HistGradientBoosting

O Target Encoding transforma cada categoria em estatísticas calculadas no conjunto de treino. Em seguida, o HistGradientBoosting aprende relações não lineares sem impor diretamente uma ordem aos códigos dos aeroportos.


In [ ]:
preprocessador_hgb = ColumnTransformer([
    ("categoricas", Pipeline([
        ("imputacao", SimpleImputer(strategy="most_frequent")),
        ("target_encoding", TargetEncoder(
            target_type="multiclass",
            smooth=20.0,
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        )),
    ]), categoricas),
    ("numericas", SimpleImputer(strategy="median"), numericas),
])
modelo_hgb = Pipeline([
    ("preprocessamento", preprocessador_hgb),
    ("classificador", HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.08,
        max_leaf_nodes=31,
        min_samples_leaf=50,
        l2_regularization=1.0,
        class_weight="balanced",
        random_state=42,
    )),
])


## 17. Treinar e avaliar o HistGradientBoosting

O Target Encoder é ajustado dentro do pipeline usando somente os dados de treino. Isso evita que a distribuição do teste influencie as variáveis de entrada.


In [ ]:
print("Rodando o modelo.fit(). Isso pode levar alguns minutos")
modelo_hgb.fit(X_treino, y_treino)
pred_hgb = modelo_hgb.predict(X_teste)
print(classification_report(
    y_teste,
    pred_hgb,
    labels=list(FAIXAS),
    target_names=list(FAIXAS.values()),
    zero_division=0,
))
print({
    "accuracy": accuracy_score(y_teste, pred_hgb),
    "balanced_accuracy": balanced_accuracy_score(y_teste, pred_hgb),
    "macro_f1": f1_score(y_teste, pred_hgb, average="macro"),
})
mostrar_matrizes(y_teste, pred_hgb, "hist gradient boosting")


## 18. Comparar os três modelos

A tabela final permite escolher o modelo de referência de acordo com a métrica prioritária. O Random Forest tem o melhor macro-F1 observado, enquanto o HistGradientBoosting tem a melhor balanced accuracy.


In [ ]:
metricas_completas = pd.DataFrame({
    "modelo": ["baseline", "regressao_logistica", "random_forest", "hist_gradient_boosting"],
    "balanced_accuracy_teste": [
        balanced_accuracy_score(y_teste, pred_baseline),
        balanced_accuracy_score(y_teste, pred_teste),
        balanced_accuracy_score(y_teste, pred_rf),
        balanced_accuracy_score(y_teste, pred_hgb),
    ],
    "macro_f1_teste": [
        f1_score(y_teste, pred_baseline, average="macro"),
        f1_score(y_teste, pred_teste, average="macro"),
        f1_score(y_teste, pred_rf, average="macro"),
        f1_score(y_teste, pred_hgb, average="macro"),
    ],
})
metricas_completas


## Conclusão provisória

Este notebook estabelece a primeira comparação entre um baseline simples e um modelo multiclasses. Os próximos experimentos devem testar pesos de classe, modelos baseados em árvores, calibração das probabilidades e a versão binária com limite de 15 minutos.

Os resultados não devem ser interpretados como relação causal. Eles medem desempenho preditivo em uma janela temporal futura.

### O que seriam valores bons de balanced accuracy e macro F1?

Não existe um limite universal, mas para este problema com seis classes e forte desbalanceamento eu usaria esta referência:

| Métrica | Limitado | Útil | Bom | Forte |
|---|---:|---:|---:|---:|
| Balanced accuracy | `< 0,25` | `0,25–0,40` | `0,40–0,60` | `> 0,60` |
| Macro-F1 | `< 0,20` | `0,20–0,35` | `0,35–0,55` | `> 0,55` |

Uma interpretação prática:

- `balanced_accuracy = 0,1667`: desempenho equivalente ao baseline de uma classe entre seis.
- `0,30`: já indica aprendizado, mas ainda limitado.
- `0,40–0,50`: desempenho potencialmente útil.
- `> 0,60`: começa a ser forte, desde que nenhuma classe esteja sendo ignorada.
- `> 0,70`: seria bastante forte para este problema.

Para o macro-F1, eu consideraria `0,50` um resultado bom e `0,60` ou mais forte, porque essa métrica exige equilíbrio entre precisão e revocação em todas as classes.

Além da média, é importante verificar:

- F1 de cada faixa;
- recall das classes de atraso elevado;
- matriz de confusão;
- estabilidade entre validação e teste.

Por exemplo, um modelo com macro-F1 `0,55`, mas F1 de apenas `0,05` para atrasos superiores a 60 minutos, não seria realmente forte para o objetivo do projeto.